In [9]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, model_validator

In [10]:
model = ChatOpenAI( 
    # model="unsloth/gemma-4-e2b-it",
    model="qwen3-30b-a3b-instruct-2507",
    base_url="http://...:.../v1",
    api_key="lm-studio",
)

In [11]:
# 원하는 데이터 구조 정의
class FinancialAdvice(BaseModel):
    setup: str = Field(description="금융 조언 상황을 설정하기 위한 질문")
    advice: str= Field(description="질문을 해결하기 위한 금융 답변")
    # Pydantic을 ㅏ용한 사용자 정의 검증 로직
    @model_validator(mode="before")
    @classmethod
    def question_ends_with_question_mark(cls, values: dict) -> dict:
        setup = values.get("setup", "")
        if not setup.endswith("?"):
            raise ValueError("잘못된 질문 형식입니다! 질문은 '?'로 끝나야 합니다.")
        return values

In [12]:
# 파서 설정 및 프롬프트 템플릿에 지침 삽입
parser = PydanticOutputParser(pydantic_object=FinancialAdvice)
prompt = PromptTemplate(
    template="다음 금융 질문 관련에 답변해 주세요.\n{format_instructions}\n질문: {query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 언어 모델을 사용해 데이터 구조를 채우도록 프롬프트와 모델 설정
chain = prompt | model | parser

In [14]:
# 체인 실행 및 결과 출력
try:
    result = chain.invoke({"query":"부동산에 관련하여 금융 조언을 받을 수 있게 질문하라."})
    print(result)
except Exception as e :
    print(f"오류 발생: {e}")

오류 발생: Failed to parse FinancialAdvice from completion {"setup": "부동산 투자에 관심이 있으며, 초기 자금의 규모와 예상 수익률, 그리고 리스크 관리 방안에 대해 고민하고 있습니다.", "advice": "부동산 투자 시 초기 자금은 반드시 전체 비용의 20~30% 이상을 확보하는 것이 좋습니다. 또한, 대출 이용 시 이자율과 상환 기간을 체크하고, 수익률 예상은 실제 임대수입과 인프라 개발 등 외부 요인을 고려해 현실적으로 설정해야 합니다. 리스크 관리로는 다변화 투자(다른 지역이나 자산 유형)와 비상금 확보가 중요하며, 장기 보유를 전제로 한 안정적인 수익 모델을 설계하는 것이 바람직합니다."}. Got: 1 validation error for FinancialAdvice
  Value error, 잘못된 질문 형식입니다! 질문은 '?'로 끝나야 합니다. [type=value_error, input_value={'setup': '부동산 투... 바람직합니다.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


In [16]:
# 구조 수정
prompt = PromptTemplate(
    template="""
        다음 요청에 맞는 금융 상담 질문과 답변을 생성하세요.

        조건:
        - setup은 사용자가 실제로 물어보는 질문문이어야 합니다.
        - setup은 반드시 '?'로 끝나야 합니다.
        - setup은 상황 설명 문장이면 안 됩니다.
        - advice는 setup 질문에 대한 답변이어야 합니다.

        {format_instructions}

        요청: {query}
    """,
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

In [18]:
prompt

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"setup": {"description": "금융 조언 상황을 설정하기 위한 질문", "title": "Setup", "type": "string"}, "advice": {"description": "질문을 해결하기 위한 금융 답변", "title": "Advice", "type": "string"}}, "required": ["setup", "advice"]}\n```'}, template="\n        다음 요청에 맞는 금융 상담 질문과 답변을 생성하세요.\n\n        조건:\n        - setup은 사용자가 실제로 물어보는 질문문이어야 합니다.\n        - setup은 반드시 '?'로 끝나야 합니다.\n        - setup은 상황 설명 문장이면 안 됩니다.\n        - advice는 setup 질문에 대

In [17]:
# 체인 실행 및 결과 출력
try:
    result = chain.invoke({
        "query": "부동산 투자에 대해 금융 조언을 받기 위한 질문 하나와 그에 대한 답변을 생성하라."
    })
    print(result)
except Exception as e :
    print(f"오류 발생: {e}")

setup='부동산 투자에 관심이 있지만, 초기 자금은 제한적이며 장기적인 수익성을 어떻게 확보할 수 있을지 고민 중입니다. 어떤 전략을 추천하시나요?' advice='초기 자금이 부족하다면, 소규모 부동산 투자 또는 리모델링 후 전세로 전환하는 방식으로 시작하는 것이 좋습니다. 또한, 장기적으로는 임대 수익률과 자산가치 증가를 고려한 지역 선택이 핵심입니다. 추가로, 모기지 대출의 이자율 변동을 고려해 고정금리 상품을 활용하고, 세제 혜택(예: 주택임대소득 감면)도 적극적으로 이용하세요.'


In [19]:
query = "부동산 투자에 대한 금융 조언을 해줘"

formatted_prompt = prompt.format(query=query)

print(formatted_prompt)


        다음 요청에 맞는 금융 상담 질문과 답변을 생성하세요.

        조건:
        - setup은 사용자가 실제로 물어보는 질문문이어야 합니다.
        - setup은 반드시 '?'로 끝나야 합니다.
        - setup은 상황 설명 문장이면 안 됩니다.
        - advice는 setup 질문에 대한 답변이어야 합니다.

        The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"setup": {"description": "금융 조언 상황을 설정하기 위한 질문", "title": "Setup", "type": "string"}, "advice": {"description": "질문을 해결하기 위한 금융 답변", "title": "Advice", "type": "string"}}, "required": ["setup", "advice"]}
```

        요청: 부동산 투자에 대한 금융 조언을 해줘
    
